In [1]:
from transformers import pipeline
from PIL import Image, ImageDraw
import torch
import operator
import numpy as np


from functions.cv.seasonal_images import get_mask

/Users/paulina/Desktop/GIT/color_palette/.venv_color_palette/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
img = Image.open("example_img.jpg")
mask = get_mask(img)

No model was supplied, defaulted to facebook/detr-resnet-50-panoptic and revision d53b52a.
Using a pipeline without specifying a model name and revision in production is not recommended.
Loading weights: 100%|██████████| 562/562 [00:00<00:00, 1155.35it/s, Materializing param=mask_head.output_conv.weight]                                
The image processor of type `DetrImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 
`label_ids_to_fuse` unset. No instance will be fused.


In [8]:
spring_colors = [
    "#FFD1BA",  # Peach
    "#FF6F61",  # Coral
    "#FFA07A",  # Salmon
    "#FFB6C1",  # Light Pink
    "#FFF68F",  # Buttercup Yellow
    "#98FF98",  # Mint
    "#87CEEB",  # Sky Blue
]

winter_colors = [
    "#0D1B2A",  # Navy
    "#1B263B",  # Dark Slate Blue
    "#415A77",  # Steel Blue
    "#778DA9",  # Cool Gray Blue
    "#E0E1DD",  # Icy White
    "#FF4C4C",  # Crimson Red
    "#6B2D5C",  # Deep Plum
]

summer_colors = [
    "#B0C4DE",  # Light Steel Blue
    "#AEC6CF",  # Powder Blue
    "#C1DAD6",  # Pale Teal
    "#F0E68C",  # Soft Khaki
    "#F5F5DC",  # Beige / Cream
    "#D8BFD8",  # Thistle / Soft Purple
    "#FFB7C5",  # Pale Pink
]

autumn_colors = [
    "#8B4513",  # Saddle Brown
    "#A0522D",  # Sienna
    "#D2691E",  # Chocolate Orange
    "#CD853F",  # Peru / Warm Tan
    "#FF8C00",  # Dark Orange
    "#556B2F",  # Dark Olive Green
    "#B22222",  # Firebrick Red
]

In [17]:
def create_palette(color_palet:dict)->Image:
    # Image size
    width, height = 800, 600
    # Number of colors
    num_colors = len(color_palet)

    # Create a blank image
    img = Image.new("RGB", (width, height), "#FFFFFF")  # white background
    draw = ImageDraw.Draw(img)

    # Calculate block height
    block_height = height // num_colors

    # Draw color blocks
    for i, color in enumerate(color_palet):
        top = i * block_height
        bottom = (i + 1) * block_height
        draw.rectangle([0, top, width, bottom], fill=color)

    # Optional: draw white lines between blocks for separation
    for i in range(1, num_colors):
        y = i * block_height
        draw.line([0, y, width, y], fill="#FFFFFF", width=4)

    return img


In [18]:
def prepare_output(img:Image, palette:Image, mask:Image) -> Image:

    img = img.resize(mask.size)
    palette = palette.resize(mask.size)
    mask_array = np.array(mask)
    mask_pil = Image.fromarray(mask_array, mode="L")
    palette.paste(img, (0, 0), mask_pil)

    return  palette 
    

In [47]:
season_palettes = {
    "autumn": autumn_colors,
    "spring": spring_colors,
    "summer": summer_colors,
    "winter": winter_colors
}
for cp_name, cp in season_palettes.items():
    palette = create_palette(cp)
    palette = prepare_output(img, palette, mask)
    palette.save(f"{cp_name}.jpg")


In [40]:
palette.show()